# Lecture 3 — SOLUTIONS: Sampling in time, space and populations

**Guiding idea.** Lecture 2 was about sampling **molecules**: few UMIs per cell or bin, many zeros,
single genes that cannot be trusted. Here we move up one level and ask *what* we actually measured.
Every dataset is a sample taken at several nested levels:

```
time point → organoid / batch → section → cell or bin → molecules
```

Each level adds its own variance and its own bias. The question to keep asking is:
**at which level am I replicating, and at which level do I want to generalise?**

Same way of working as in Lectures 1 and 2: read the question, complete the **decomposition tree**,
write **one function per leaf** and a root function that calls them.

| Part | Exercises | Data |
|---|---|---|
| Time | 1 — the peak that wasn't there · 2 — how old is this organoid? · 3 — pseudotime from a snapshot | simulations |
| Space | 4 — a slice of an organoid · 5 — neighbouring bins are not independent | simulation + `Datasets/visiumhd_mouse_brain_cortex_2um.npz` |
| Populations | 6 — how many cells do I need? · 7 — population traps | simulations |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats, ndimage, sparse
from scipy.sparse import csgraph
from scipy.spatial import cKDTree

rng = np.random.default_rng(2026)

## Part A — Sampling in time

Every time point is **destructive**: we never follow the same organoid over time, we compare different
organoids harvested at different ages. A time course is therefore a handful of snapshots, and what
happens *between* snapshots is invisible.

## Exercise 1 — The peak that wasn't there (simulation)

**Question:** a gene (think of *EOMES* in intermediate progenitors) has a transient peak of expression
around day 45 that lasts about two weeks. With a time course of 3, 5 or 10 time points between day 10 and
day 120, do we see the peak?

```
Can a time course see a transient peak?
├── The truth           → true_expression(t): baseline + Gaussian peak (centre 45 d, width 6 d)
├── The design          → timepoints(n, start=10, end=120): n equally spaced days
├── The measurement     → measure(days, n_organoids=3): true value + biological noise (CV 20%)
├── The summary         → observed peak height / true peak height
└── Plot                → true curve + measured points, one panel per design
```

**To do:** write the four functions and `peak_recovery(n_points, shift=0)`, which returns the
observed/true ratio for a design shifted by `shift` days. Then compute it for `n_points` in 3…15 and
for many random shifts of the schedule.

*Discussion question:* in a published time course of organoids (30, 60, 90, 120 days, a very common
design) could you claim that a gene is "never expressed"? What is the minimum spacing you need, given the
width of the peak? (hint: think of the sampling theorem.)

In [ ]:
def true_expression(t, baseline=1.0, amplitude=10.0, centre=45.0, width=6.0):
    return baseline + amplitude * np.exp(-(np.asarray(t) - centre) ** 2 / (2 * width ** 2))

def timepoints(n, start=10.0, end=120.0, shift=0.0):
    return np.linspace(start, end, n) + shift

def measure(days, n_organoids=3, cv=0.2, rng=rng):
    true = true_expression(days)
    noisy = true[:, None] * rng.lognormal(0, cv, size=(len(days), n_organoids))
    return noisy

def peak_recovery(n_points, shift=0.0):
    days = timepoints(n_points, shift=shift)
    observed = true_expression(days).max()      # best case: no noise
    return observed / true_expression(45.0)

def plot_designs(designs=(3, 5, 10)):
    t = np.linspace(0, 130, 500)
    fig, axes = plt.subplots(1, len(designs), figsize=(4 * len(designs), 3), sharey=True)
    for ax, n in zip(axes, designs):
        days = timepoints(n)
        values = measure(days)
        ax.plot(t, true_expression(t), color="grey", label="truth")
        ax.plot(np.repeat(days, values.shape[1]), values.ravel(), "o", alpha=0.6, label="organoids")
        ax.plot(days, values.mean(axis=1), "-", color="C3", label="what we report")
        ax.set_title(f"{n} time points")
        ax.set_xlabel("day")
    axes[0].set_ylabel("expression")
    axes[0].legend()
    plt.tight_layout()
    plt.show()

plot_designs()

ns = np.arange(3, 16)
shifts = rng.uniform(-5, 5, size=200)
recovery = pd.DataFrame({
    "n_points": ns,
    "spacing_days": [110 / (n - 1) for n in ns],
    "mean_recovery": [np.mean([peak_recovery(n, s) for s in shifts]) for n in ns],
    "worst_recovery": [np.min([peak_recovery(n, s) for s in shifts]) for n in ns],
}).round(2)
recovery

**Comment.** The peak is ~12 days wide at half height. With spacing ≥ 20 days the observed maximum
depends on *luck* (where the schedule falls), and in the worst case the peak almost disappears.
A rule of thumb from the sampling theorem: spacing must be well below the width of the phenomenon you
want to see. Absence of a signal in a sparse time course is not evidence of absence.

## Exercise 2 — How old is this organoid? (simulation)

**Question:** organoids develop more slowly than the fetal cortex, and not all at the same speed.
If we align "day 60 of an organoid" to a fixed post-conception week, how wrong can we be?
Can we estimate the biological age from the transcriptome instead (the time-wise benchmarking against
BrainSpan of Nico's Lecture 2)?

The simulated reference has 5 gene modules of 10 genes each, with different dynamics between post-conception
week (PCW) 8 and 24: progenitors (decreasing), intermediate progenitors (early peak), deep-layer neurons
(middle peak), upper-layer neurons (late rise), glia (late rise).

```
Biological age of an organoid
├── Reference         → reference_profile(pcw): 50 genes, smooth in PCW
├── Organoids         → simulate_organoid(day, speed): biological PCW = 6 + speed * day / 7, plus noise
├── Naive alignment   → calendar_pcw(day) = 6 + day / 7  (assumes speed = 1)
├── Transcriptomic    → estimate_pcw(profile): PCW of the reference with the highest correlation
└── Compare           → error of the two estimates vs the true PCW
```

**To do:** write the functions, simulate 40 organoids (days 30–120, speed uniform in 0.5–1.0) and compare
the errors of the two approaches.

*Discussion question:* the transcriptomic estimate is better, but it uses the same genes that you might
then want to compare between organoid and fetus. What is circular about this? And what if the organoid is
not "a slower fetus" but follows a different trajectory altogether?

In [ ]:
MODULES = {
    "progenitors":   lambda w: 1 / (1 + np.exp((w - 14) / 2)),
    "intermediate":  lambda w: np.exp(-(w - 12) ** 2 / (2 * 2.5 ** 2)),
    "deep_layer":    lambda w: np.exp(-(w - 16) ** 2 / (2 * 3 ** 2)),
    "upper_layer":   lambda w: 1 / (1 + np.exp(-(w - 19) / 1.5)),
    "glia":          lambda w: 1 / (1 + np.exp(-(w - 22) / 1.5)),
}
GENES_PER_MODULE = 10
gene_weights = rng.uniform(0.5, 1.5, size=(len(MODULES), GENES_PER_MODULE))
REFERENCE_PCW = np.arange(8, 24.5, 0.5)

def reference_profile(pcw):
    module_levels = np.array([f(pcw) for f in MODULES.values()])          # 5
    return (module_levels[:, None] * gene_weights).ravel() * 8 + 1        # 50 genes, log-scale-ish

def simulate_organoid(day, speed, noise=0.8, rng=rng):
    true_pcw = 6 + speed * day / 7
    profile = reference_profile(true_pcw) + rng.normal(0, noise, size=len(MODULES) * GENES_PER_MODULE)
    return profile, true_pcw

def calendar_pcw(day):
    return 6 + day / 7

def estimate_pcw(profile):
    reference = np.array([reference_profile(w) for w in REFERENCE_PCW])
    r = [np.corrcoef(profile, ref)[0, 1] for ref in reference]
    return REFERENCE_PCW[int(np.argmax(r))]

def compare_alignments(n=40):
    rows = []
    for _ in range(n):
        day = rng.uniform(30, 120)
        speed = rng.uniform(0.5, 1.0)
        profile, true_pcw = simulate_organoid(day, speed)
        rows.append({"day": day, "speed": speed, "true_pcw": true_pcw,
                     "calendar_pcw": calendar_pcw(day), "transcriptomic_pcw": estimate_pcw(profile)})
    return pd.DataFrame(rows)

aligned = compare_alignments()
errors = pd.DataFrame({
    "calendar": (aligned["calendar_pcw"] - aligned["true_pcw"]).abs(),
    "transcriptomic": (aligned["transcriptomic_pcw"] - aligned["true_pcw"]).abs(),
})
display(errors.describe().round(2))

plt.figure(figsize=(5, 4))
plt.scatter(aligned["true_pcw"], aligned["calendar_pcw"], label="calendar")
plt.scatter(aligned["true_pcw"], aligned["transcriptomic_pcw"], label="transcriptomic")
plt.plot([8, 24], [8, 24], "k--", lw=1)
plt.xlabel("true biological age (PCW)")
plt.ylabel("estimated age (PCW)")
plt.legend()
plt.show()

## Exercise 3 — Pseudotime from a snapshot (simulation)

**Question:** trajectory methods (Nico's Lecture 3) order cells collected *at the same moment* along a
continuous path. When does this work, and when does it invent an order that is not there?

We simulate cells along a curved differentiation path (almost a full circle in a 2D latent space, so that
the two ends are not far apart in expression space, embedded in 30 genes with noise). The true time `t` goes from 0 (progenitor) to 1 (neuron).

```
Pseudotime from a snapshot
├── Simulate        → simulate_trajectory(n, gap=None): cells at t ~ U(0,1), optionally without t in gap;
│                      latent position (cos 1.8πt, sin 1.8πt), times a random 2×30 matrix, plus noise
├── Method 1        → pc1_pseudotime(X): position along the first principal component
├── Method 2        → graph_pseudotime(X, root, k): shortest-path distance on the k-nearest-neighbour graph
└── Evaluate        → Spearman correlation between pseudotime and true t (only for reachable cells)
```

**To do:** compare the two methods on (i) a dense snapshot and (ii) a snapshot where the intermediate
states (`0.45 < t < 0.6`) were not sampled — for example because they are rare, or because they exist only
for a few days between two harvests.

*Discussion question:* with the gap, the graph is either disconnected (some cells are unreachable) or, if
you increase `k`, it connects through a shortcut. Which of the two failures is more dangerous, and why?
Who decides which end of the trajectory is the "root"?

In [ ]:
N_GENES_TRAJ = 30
embedding = rng.normal(size=(2, N_GENES_TRAJ))

def simulate_trajectory(n=600, gap=None, noise=0.25, rng=rng):
    t = rng.uniform(0, 1, size=4 * n)
    if gap is not None:
        t = t[(t < gap[0]) | (t > gap[1])]
    t = np.sort(t[:n])
    angle = 1.8 * np.pi * t
    latent = np.column_stack([np.cos(angle), np.sin(angle)])
    X = latent @ embedding + rng.normal(0, noise, size=(len(t), N_GENES_TRAJ))
    return X, t

def pc1_pseudotime(X):
    Xc = X - X.mean(axis=0)
    _, _, vt = np.linalg.svd(Xc, full_matrices=False)
    return Xc @ vt[0]

def graph_pseudotime(X, root, k=10):
    dist, idx = cKDTree(X).query(X, k=k + 1)
    n = len(X)
    rows = np.repeat(np.arange(n), k)
    graph = sparse.csr_matrix((dist[:, 1:].ravel(), (rows, idx[:, 1:].ravel())), shape=(n, n))
    return csgraph.dijkstra(graph, directed=False, indices=root)

def evaluate(pseudotime, t):
    ok = np.isfinite(pseudotime)
    return {"abs_spearman_vs_true_t": round(abs(stats.spearmanr(pseudotime[ok], t[ok])[0]), 3),
            "reachable_cells": f"{ok.sum()}/{len(t)}"}

def run_case(name, gap=None, k=10):
    X, t = simulate_trajectory(gap=gap)
    root = int(np.argmin(t))
    return [{"case": name, "method": "PC1", **evaluate(pc1_pseudotime(X), t)},
            {"case": name, "method": f"graph, k={k}", **evaluate(graph_pseudotime(X, root, k), t)}]

display(pd.DataFrame(run_case("dense") + run_case("gap", gap=(0.45, 0.6), k=10)
                     + run_case("gap", gap=(0.45, 0.6), k=60)))

X, t = simulate_trajectory()
pc = pc1_pseudotime(X)
plt.figure(figsize=(5, 3))
plt.scatter(t, pc, s=5)
plt.xlabel("true time t")
plt.ylabel("PC1 pseudotime")
plt.title("PC1 folds a curved trajectory")
plt.show()

## Part B — Sampling in space

## Exercise 4 — A slice of an organoid (simulation)

**Question:** an organoid is a 3D object; a section for histology or spatial transcriptomics is a 2D
slice of it. If an organoid of radius 1 mm contains 10 ventricular-like rosettes (radius 100 µm),
how many rosettes do we see in one section? How often do we see none?

```
Rosettes in one section
├── Organoid        → make_organoid(n_rosettes): random rosette centres inside the sphere (radius 800 µm)
├── Section         → rosettes_in_section(centres, z, r=100): rosettes with |z_centre − z| < r
├── Many sections   → section_survey(n_organoids, sections): counts for random organoids and section heights
└── Summary         → distribution of counts, P(0 rosettes), central vs peripheral sections
```

**To do:** write the functions and compare sections taken at random heights with sections taken
near the centre (|z| < 200 µm), which is what you usually try to do.

*Discussion question:* "organoids under condition X have fewer rosettes" — measured on one section per
organoid. What else could explain the result? (think of organoid size and of where you cut.)
What does stereology recommend?

In [ ]:
ORGANOID_RADIUS = 1000.0
ROSETTE_RADIUS = 100.0

def make_organoid(n_rosettes=10, max_radius=800.0, rng=rng):
    centres = []
    while len(centres) < n_rosettes:
        p = rng.uniform(-max_radius, max_radius, size=3)
        if np.linalg.norm(p) < max_radius:
            centres.append(p)
    return np.array(centres)

def rosettes_in_section(centres, z, r=ROSETTE_RADIUS):
    return int(np.sum(np.abs(centres[:, 2] - z) < r))

def section_survey(n_organoids=2000, central=False, rng=rng):
    counts = []
    for _ in range(n_organoids):
        centres = make_organoid()
        z = rng.uniform(-200, 200) if central else rng.uniform(-ORGANOID_RADIUS * 0.9, ORGANOID_RADIUS * 0.9)
        counts.append(rosettes_in_section(centres, z))
    return np.array(counts)

survey = {"random height": section_survey(), "central section": section_survey(central=True)}
summary = pd.DataFrame({
    name: {"mean": c.mean(), "sd": c.std(), "P(0 rosettes)": np.mean(c == 0), "max": c.max()}
    for name, c in survey.items()
}).T.round(2)
display(summary)

fig, ax = plt.subplots(figsize=(5, 3))
bins = np.arange(0, 11) - 0.5
for name, c in survey.items():
    ax.hist(c, bins=bins, alpha=0.6, density=True, label=name)
ax.set_xlabel("rosettes visible in the section (true number = 10)")
ax.set_ylabel("fraction of organoids")
ax.legend()
plt.show()

## Exercise 5 — Neighbouring bins are not independent

**Question:** a Visium HD section of the mouse brain at 16 µm gives ~100,000 bins. Can we treat them as 100,000 independent
observations in a statistical test?

### 5a — Simulation: two halves of the *same* tissue

We simulate a smooth random field (spatially autocorrelated noise, **no difference** between left and right)
and test "left vs right" with a t-test, first using every bin as an observation, then using blocks of
bins as the unit.

```
False positives with spatial data
├── Field          → smooth_field(shape, smoothness): white noise + Gaussian filter, standardised
├── Bin-level test → t-test left vs right using every bin
├── Block test     → average blocks of 20×20 bins, t-test on block means
└── Repeat         → fraction of p < 0.05 over 200 fields, for both tests
```

*The level of the null hypothesis is always 5%: anything far above that is a broken test.*

In [ ]:
def smooth_field(shape=(100, 100), smoothness=5.0, rng=rng):
    field = ndimage.gaussian_filter(rng.normal(size=shape), smoothness)
    return (field - field.mean()) / field.std()

def bin_level_test(field):
    half = field.shape[1] // 2
    return stats.ttest_ind(field[:, :half].ravel(), field[:, half:].ravel()).pvalue

def block_means(field, block=20):
    h, w = field.shape
    return field[: h // block * block, : w // block * block].reshape(h // block, block, w // block, block).mean(axis=(1, 3))

def block_level_test(field, block=20):
    means = block_means(field, block)
    half = means.shape[1] // 2
    return stats.ttest_ind(means[:, :half].ravel(), means[:, half:].ravel()).pvalue

def false_positive_rates(n_fields=200, smoothness=5.0):
    fields = [smooth_field(smoothness=smoothness) for _ in range(n_fields)]
    return {"bins as units": np.mean([bin_level_test(f) < 0.05 for f in fields]),
            "20x20 blocks as units": np.mean([block_level_test(f) < 0.05 for f in fields])}

pd.DataFrame({s: false_positive_rates(smoothness=s) for s in [0.01, 2.0, 5.0]}).T.rename_axis("smoothness")

### 5b — Real data: how autocorrelated is a Visium HD gene?

`Datasets/visiumhd_mouse_brain_cortex_2um.npz` is the crop of an adult mouse brain Visium HD section used in
Lecture 2 (Exercise 4): 2 µm bins, 67 marker genes. We aggregate it to 16 µm bins.

**Moran's I** measures spatial autocorrelation: ≈ 0 for spatially random values, close to 1 when neighbours
are very similar.

$$I = \frac{N}{W} \frac{\sum_i \sum_j w_{ij}(x_i-\bar x)(x_j-\bar x)}{\sum_i (x_i-\bar x)^2}$$

```
Moran's I of each gene
├── Load         → counts (bins × genes), bins (row, col)
├── Normalise    → log1p(counts / total × median total)
├── Neighbours   → rook neighbours on the grid (up, down, left, right) as a sparse matrix W
├── Moran's I    → morans_i(x, W)
└── Control      → the same gene with bins shuffled in space
```

*Discussion question:* if a gene has Moran's I = 0.6, roughly how many "independent" observations do
100,000 bins correspond to? Is the answer the same for every gene?

In [ ]:
def load_visium(path, size_um=16):
    # load the 2 µm crop and sum it into size_um × size_um bins (same as Lecture 2, Exercise 4)
    z = np.load(path)
    counts = sparse.csr_matrix((z["data"], z["indices"], z["indptr"]), shape=tuple(z["shape"]))
    b = size_um // 2
    key = (z["row"].astype(int) // b) * 100_000 + z["col"].astype(int) // b
    unique, inverse = np.unique(key, return_inverse=True)
    assign = sparse.csr_matrix((np.ones(len(key)), (inverse, np.arange(len(key)))), shape=(len(unique), len(key)))
    bins = pd.DataFrame({"array_row": unique // 100_000, "array_col": unique % 100_000,
                         "total_counts": np.bincount(inverse, weights=z["total_counts"])})
    return (assign @ counts).tocsr(), bins, list(z["genes"])

counts, bins, genes = load_visium("../Datasets/visiumhd_mouse_brain_cortex_2um.npz")
print("16 µm bins:", counts.shape)

def normalise(counts, totals):
    scale = np.median(totals) / totals
    return np.log1p(counts.multiply(scale[:, None]).toarray())

def rook_neighbours(rows, cols):
    index = {(r, c): i for i, (r, c) in enumerate(zip(rows, cols))}
    pairs = [(i, index[(r + dr, c + dc)])
             for i, (r, c) in enumerate(zip(rows, cols))
             for dr, dc in [(1, 0), (-1, 0), (0, 1), (0, -1)]
             if (r + dr, c + dc) in index]
    i, j = np.array(pairs).T
    return sparse.csr_matrix((np.ones(len(i)), (i, j)), shape=(len(rows), len(rows)))

def morans_i(x, W):
    z = x - x.mean()
    return len(x) / W.sum() * (z @ (W @ z)) / (z @ z)

expr = normalise(counts, bins["total_counts"].to_numpy())
W = rook_neighbours(bins["array_row"].to_numpy(), bins["array_col"].to_numpy())
moran = pd.DataFrame({
    "gene": genes,
    "mean_counts": np.asarray(counts.mean(axis=0)).ravel(),
    "morans_I": [morans_i(expr[:, g], W) for g in range(len(genes))],
    "morans_I_shuffled": [morans_i(rng.permutation(expr[:, g]), W) for g in range(len(genes))],
}).sort_values("morans_I", ascending=False).round(3)
moran

**Comment.** Highly expressed, anatomically patterned genes have high Moran's I; after shuffling, I ≈ 0.
With strong autocorrelation the *effective* sample size is much smaller than the number of bins, so a
test with n = bins gives absurdly small p-values (5a). The fix is the same as with cells in Lecture 2
(6c): test at the level of the real replicate (blocks, regions, sections, animals).

## Part C — Sampling populations

## Exercise 6 — How many cells do I need? (analytic + simulation)

**Question:** a population is present at frequency *f* (e.g. outer radial glia at 1%, Cajal–Retzius cells
at 0.2%). How many cells must I sequence to see at least *k* of them with probability *p*?

For *k* = 1 there is a closed formula:
$$P(\text{at least one}) = 1-(1-f)^n \ge p \;\Rightarrow\; n \ge \frac{\ln(1-p)}{\ln(1-f)}$$
To be recognised as a cluster, though, a population needs tens of cells, not one.

```
Cells needed
├── Closed formula     → n_needed_one(f, p)
├── General k          → n_needed(f, p, k): smallest n with P(Binomial(n, f) ≥ k) ≥ p
├── Table              → f ∈ {5%, 1%, 0.5%, 0.1%} × k ∈ {1, 10, 30}, p = 0.95
└── Check by simulation→ draw n cells many times, fraction of draws with ≥ k cells of the population
```

*Discussion question:* the table assumes every cell is captured with the same probability. In Exercise 7d
we will see that this is false. How does that change the numbers? And if the population exists only in 2
of 6 organoids?

In [ ]:
def n_needed_one(f, p=0.95):
    return int(np.ceil(np.log(1 - p) / np.log(1 - f)))

def n_needed(f, p=0.95, k=1):
    n = k
    step = max(1, int(k / f / 1000))
    while stats.binom.sf(k - 1, n, f) < p:
        n += step
    while n > k and stats.binom.sf(k - 1, n - 1, f) >= p:   # step back to the exact minimum
        n -= 1
    return n

def check_by_simulation(f, n, k, n_draws=5000, rng=rng):
    return np.mean(rng.binomial(n, f, size=n_draws) >= k)

frequencies = [0.05, 0.01, 0.005, 0.001]
table = pd.DataFrame({f"k = {k}": [n_needed(f, k=k) for f in frequencies] for k in [1, 10, 30]},
                     index=[f"f = {f:.1%}" for f in frequencies])
table["closed formula (k=1)"] = [n_needed_one(f) for f in frequencies]
display(table)
print("simulation check, f = 0.5%, k = 10:", check_by_simulation(0.005, table.loc["f = 0.5%", "k = 10"], 10))

## Exercise 7 — Population traps

Four mini-cases, in the spirit of Exercise 6 of Lecture 1.

| Part | Trap |
|---|---|
| 7a | compositional data: one population grows, all the others seem to shrink |
| 7b | cells vs organoids: the unit of replication for proportions |
| 7c | batch confounded with condition |
| 7d | dissociation bias: the proportions in the dataset are not those in the tissue |

### 7a — One population grows, all the others "shrink"

Four cell types; in condition B **only** type A doubles in absolute number, the other three are unchanged.
6 organoids per condition, 3000 cells sequenced per organoid (always the same number: the machine does not
know how big the organoid was).

```
Compositional effect
├── Simulate      → absolute cell numbers per organoid (log-normal noise), then sample 3000 cells
├── Proportions   → per organoid
├── Test          → t-test of each type's proportion, A vs B
└── Ratio         → test log(type / reference type D) instead, knowing that D did not change
```

*Discussion question:* without external information (absolute counts, spike-in cells, a population
known not to change) can you tell "A increased" from "B, C, D decreased"?

In [ ]:
CELL_TYPES = ["A", "B", "C", "D"]
BASE_ABUNDANCE = np.array([1000, 3000, 4000, 2000])

def simulate_composition(fold_a=1.0, n_organoids=6, cells_sequenced=3000, rng=rng):
    abundance = BASE_ABUNDANCE * np.array([fold_a, 1, 1, 1])
    absolute = abundance * rng.lognormal(0, 0.15, size=(n_organoids, len(CELL_TYPES)))
    sequenced = np.array([rng.multinomial(cells_sequenced, a / a.sum()) for a in absolute])
    return pd.DataFrame(sequenced, columns=CELL_TYPES)

def proportions(counts):
    return counts.div(counts.sum(axis=1), axis=0)

def compare_conditions(control, treated):
    pc, pt = proportions(control), proportions(treated)
    rows = []
    for ct in CELL_TYPES:
        rows.append({"cell_type": ct,
                     "prop_control": pc[ct].mean(), "prop_treated": pt[ct].mean(),
                     "p_proportion": stats.ttest_ind(pc[ct], pt[ct]).pvalue,
                     "p_ratio_to_D": stats.ttest_ind(np.log(control[ct] / control["D"]),
                                                     np.log(treated[ct] / treated["D"])).pvalue})
    return pd.DataFrame(rows).set_index("cell_type")

compare_conditions(simulate_composition(1.0), simulate_composition(2.0)).style.format("{:.3g}")

### 7b — Cells or organoids?

The fraction of a cell type varies between organoids (organoid-to-organoid variability), but there is
**no** difference between conditions. We compare two tests, repeated over 500 simulated experiments:

```
Unit of replication for proportions
├── Simulate      → 4 organoids per condition, proportion ~ Beta(mean 20%, between-organoid sd ≈ 5%), 2000 cells each
├── Cells as n    → chi-square test on the 2×2 table of pooled cell counts
├── Organoids as n→ t-test on the 4 + 4 per-organoid proportions
└── Repeat        → false positive rate of each test
```

*Discussion question:* the pooled test has 16,000 "observations". How many does it really have?

In [ ]:
def simulate_fractions(n_organoids=4, cells=2000, mean=0.2, concentration=60, rng=rng):
    p = rng.beta(mean * concentration, (1 - mean) * concentration, size=n_organoids)
    return rng.binomial(cells, p), np.full(n_organoids, cells)

def pooled_cell_test(k1, n1, k2, n2):
    table = [[k1.sum(), n1.sum() - k1.sum()], [k2.sum(), n2.sum() - k2.sum()]]
    return stats.chi2_contingency(table)[1]

def organoid_test(k1, n1, k2, n2):
    return stats.ttest_ind(k1 / n1, k2 / n2).pvalue

def false_positives(n_experiments=500):
    hits = {"cells as n": 0, "organoids as n": 0}
    for _ in range(n_experiments):
        k1, n1 = simulate_fractions()
        k2, n2 = simulate_fractions()
        hits["cells as n"] += pooled_cell_test(k1, n1, k2, n2) < 0.05
        hits["organoids as n"] += organoid_test(k1, n1, k2, n2) < 0.05
    return {name: h / n_experiments for name, h in hits.items()}

false_positives()

### 7c — Batch confounded with condition

All control organoids were differentiated in January, all treated ones in March. The treatment does
**nothing**; the March batch has a +1 shift in the measured quantity.

```
Confounded vs balanced design
├── Simulate      → simulate_design(design): 8 organoids, batch effect +1, treatment effect 0
├── Confounded    → control = batch 1, treated = batch 2
├── Balanced      → each batch has 2 controls and 2 treated
└── Estimate      → least squares with columns [intercept, treated, batch] (when possible)
```

*Discussion question:* in the confounded design, why does `np.linalg.lstsq` not give you a separate
treatment effect? Can any analysis rescue that experiment?

In [ ]:
def simulate_design(design, batch_effect=1.0, treatment_effect=0.0, rng=rng):
    if design == "confounded":
        treated = np.array([0, 0, 0, 0, 1, 1, 1, 1])
        batch = np.array([0, 0, 0, 0, 1, 1, 1, 1])
    else:
        treated = np.array([0, 0, 1, 1, 0, 0, 1, 1])
        batch = np.array([0, 0, 0, 0, 1, 1, 1, 1])
    y = treatment_effect * treated + batch_effect * batch + rng.normal(0, 0.3, size=8)
    return pd.DataFrame({"treated": treated, "batch": batch, "y": y})

def naive_difference(df):
    return df.loc[df.treated == 1, "y"].mean() - df.loc[df.treated == 0, "y"].mean()

def adjusted_estimate(df):
    X = np.column_stack([np.ones(len(df)), df["treated"], df["batch"]])
    coef, _, rank, _ = np.linalg.lstsq(X, df["y"], rcond=None)
    return coef[1] if rank == X.shape[1] else np.nan   # rank-deficient: not identifiable

pd.DataFrame({d: {"naive treated − control": naive_difference(df := simulate_design(d)),
                  "adjusted for batch": adjusted_estimate(df),
                  "rank of [1, treated, batch]": np.linalg.matrix_rank(df[["treated", "batch"]].assign(c=1))}
              for d in ["confounded", "balanced"]}).T.round(2)

### 7d — Where did the neurons go?

Dissociation kills or loses fragile cells (mature neurons with long processes) more than compact ones
(progenitors). Suppose the true composition of the tissue and the probability that a cell of each type
survives dissociation and capture are:

| type | tissue | capture |
|---|---|---|
| progenitors | 30% | 0.8 |
| intermediate progenitors | 10% | 0.7 |
| neurons | 50% | 0.3 |
| glia | 10% | 0.6 |

```
Dissociation bias
├── Observed     → expected_observed(tissue, capture): tissue × capture, renormalised
├── Simulate     → sample 5000 cells from the tissue, keep each with its capture probability
└── Correct      → divide observed by capture and renormalise (needs capture to be known!)
```

*Discussion question:* how would you estimate the capture probabilities in practice? (hint: spatial
transcriptomics or imaging on the same tissue, single-nucleus vs single-cell.)

In [ ]:
tissue = pd.Series({"progenitors": 0.30, "intermediate": 0.10, "neurons": 0.50, "glia": 0.10})
capture = pd.Series({"progenitors": 0.8, "intermediate": 0.7, "neurons": 0.3, "glia": 0.6})

def expected_observed(tissue, capture):
    raw = tissue * capture
    return raw / raw.sum()

def simulate_capture(tissue, capture, n_cells=5000, rng=rng):
    types = rng.choice(tissue.index, size=n_cells, p=tissue.values)
    kept = rng.random(n_cells) < capture[types].values
    return pd.Series(types[kept]).value_counts(normalize=True).reindex(tissue.index)

def correct(observed, capture):
    raw = observed / capture
    return raw / raw.sum()

observed = simulate_capture(tissue, capture)
pd.DataFrame({"tissue": tissue, "expected observed": expected_observed(tissue, capture),
              "simulated observed": observed, "corrected": correct(observed, capture)}).round(3)

## Wrap-up

| Level | What goes wrong | What to do |
|---|---|---|
| time | transient events fall between time points; calendar age ≠ biological age; pseudotime invents order across gaps | dense sampling around the events of interest; transcriptomic age; check connectivity |
| space | one section is a lottery; neighbouring bins are not independent | several sections at defined depths; test on blocks/regions/animals |
| populations | rare types need many cells; proportions are compositional; cells are not replicates; capture is biased | power for cell number; ratios or absolute references; organoids/batches as n; balanced designs |

**References.**
Svensson (2020) *Droplet scRNA-seq is not zero-inflated*, Nat Biotech ·
Squair et al. (2021) *Confronting false discoveries in single-cell differential expression*, Nat Commun ·
Büttner et al. (2021) *scCODA: a Bayesian model for compositional single-cell data analysis*, Nat Commun ·
Phipson et al. (2022) *propeller: testing for differences in cell type proportions*, Bioinformatics ·
Saelens et al. (2019) *A comparison of single-cell trajectory inference methods*, Nat Biotech ·
Velasco et al. (2019) *Individual brain organoids reproducibly form cell diversity of the human cerebral cortex*, Nature.